# 15. Kritisk värdering av slutsatser

Du kan nu räkna ut medelvärden, konfidensintervall, p-värden och regressionslinjer. Den sista färdigheten är den viktigaste:
att bedöma om en statistisk slutsats **håller** – andras och dina egna.

Den här notebooken samlar kursens varningar till en checklista, låter dig uppleva de vanligaste fällorna i simuleringar och
avslutas med en uppgift där du granskar ett påstående med hela verktygslådan.

**Efter den här notebooken kan du**
- ställa rätt frågor till ett statistiskt resultat
- förklara varför många tester ger falska larm
- skilja på statistisk signifikans och praktisk betydelse
- genomskåda ett samband som beror på en bakomliggande variabel

**Förkunskaper:** hela kursen, särskilt [6. Datakvalitet och bias](../06_datakvalitet_och_bias/06_datakvalitet_och_bias.ipynb),
[10. Typ I- och typ II-fel](../10_typ1_typ2_fel/10_typ1_typ2_fel.ipynb) och [12. Korrelation](../12_korrelation/12_korrelation.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

## 15.1 Checklistan

| Fråga | Varför | Notebook |
|---|---|---|
| **Hur togs stickprovet fram?** Slumpmässigt? Över- eller undertäckning? | Utan slumpmässigt urval gäller inga felmarginaler | 5, 6 |
| **Hur stort var bortfallet,** och kan det vara systematiskt? | Bias försvinner inte med större stickprov | 6 |
| **Hur stor är osäkerheten?** Redovisas konfidensintervall eller felmarginal? | En punktskattning utan osäkerhet går inte att värdera | 7 |
| **Är metodens förutsättningar uppfyllda?** | Tillräckligt stora stickprov, oberoende observationer | 7–11 |
| **Hur stor är effekten?** | Signifikant betyder inte viktigt | 9 |
| **Kan stickprovet ha varit för litet?** | Icke-signifikant betyder inte "ingen skillnad" | 10 |
| **Hur många tester gjordes?** | Många tester ger falska larm | 10, 15 |
| **Kan en tredje variabel förklara sambandet?** | Korrelation är inte kausalitet | 12 |
| **Gäller slutsatsen utanför det som undersöktes?** | Extrapolera inte | 13 |

De tre första handlar om **datan**, de övriga om **tolkningen**. Resten av notebooken fördjupar fyra av dem.

## 15.2 Fälla 1: Många tester

Från [notebook 10](../10_typ1_typ2_fel/10_typ1_typ2_fel.ipynb): när $H_0$ är sann blir ändå 5 % av testerna signifikanta.
Vad betyder det för den som gör många tester?

Nedan görs 20 t-tester. I **alla** är $H_0$ sann: båda grupperna dras ur samma fördelning.

In [ ]:
num_tests = 20          # ← ändra: pröva 5 och 100

a = rng.normal(0, 1, (num_tests, 30))
b = rng.normal(0, 1, (num_tests, 30))
p_values = stats.ttest_ind(a, b, axis=1).pvalue

print(f"{(p_values < 0.05).sum()} av {num_tests} tester blev 'signifikanta', trots att ingen verklig skillnad finns.")
print("Minsta p-värden:", np.round(np.sort(p_values)[:5], 3))

🧪 **Labba:** Kör cellen 10–15 gånger (cellen drar nya slumptal varje gång). Hur ofta får du *minst ett* signifikant resultat?

Sannolikheten går att räkna ut. Chansen att ett enskilt test *inte* larmar falskt är 0,95. Chansen att inget av 20
oberoende tester gör det är 0,95²⁰. Alltså:

$$P(\text{minst ett falskt larm}) = 1 - 0{,}95^{20} \approx 64\,\%$$

In [ ]:
for k in [1, 5, 10, 20, 50, 100]:
    print(f"{k:>3} tester: risk för minst ett falskt larm = {1 - 0.95 ** k:.0%}")

✏️ **Din tur:** Kontrollera formeln med en simulering. Gör 2 000 "studier" som var och en består av 20 tester där $H_0$ är sann.
Hur stor andel av studierna får minst ett p-värde under 0,05?

Ledtråd: `rng.normal(0, 1, (2000, 20, 30))` ger 2 000 studier × 20 tester × 30 observationer. Kör t-testet med `axis=2`
så får du en tabell med 2 000 rader och 20 p-värden per rad. `.min(axis=1)` ger det minsta p-värdet i varje studie.

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
a_sim = rng.normal(0, 1, (2000, 20, 30))
b_sim = rng.normal(0, 1, (2000, 20, 30))
p_sim = stats.ttest_ind(a_sim, b_sim, axis=2).pvalue          # form (2000, 20)
print(f"Andel studier med minst ett falskt larm: {(p_sim.min(axis=1) < 0.05).mean():.0%}")
```

</details>

### Hur det ser ut i praktiken

Ingen skriver "vi gjorde 20 tester och ett blev signifikant". Så här går det till i stället:

Ett företag påstår att deras kosttillskott förbättrar hälsan. De mäter 20 olika hälsomått på 30 personer som fått
tillskottet och 30 som fått placebo. Tillskottet är i själva verket verkningslöst.

In [ ]:
measures = ["blodtryck", "vilopuls", "kolesterol", "blodsocker", "sömnkvalitet", "reaktionstid", "minne", "humör",
            "energinivå", "greppstyrka", "kondition", "vikt", "midjemått", "hudkvalitet", "koncentration", "stressnivå",
            "aptit", "huvudvärk", "ledvärk", "immunförsvar"]

supplement = rng.normal(0, 1, (20, 30))       # 20 mått × 30 personer, ingen verklig effekt
placebo = rng.normal(0, 1, (20, 30))
p_vals = stats.ttest_ind(supplement, placebo, axis=1).pvalue

results = pd.DataFrame({"mått": measures, "p-värde": p_vals.round(3)}).sort_values("p-värde")
results.head(5)

🧪 **Labba:** Kör cellen tills du får minst ett p-värde under 0,05 (det brukar inte ta många försök). Skriv sedan rubriken
till företagets pressmeddelande.

Att leta igenom många mått, undergrupper eller analysmetoder och bara redovisa det som blev signifikant kallas **p-hackning**.
Det behöver inte vara medvetet fusk: det räcker att man "provar lite olika" och stannar när något ser intressant ut.

**Motmedel:**
- Bestäm *i förväg* vad som ska testas.
- Redovisa *alla* tester som gjorts.
- Kräv lägre p-värden när många tester görs. Den enklaste metoden (Bonferroni) är att dela α med antalet tester: 0,05 / 20 = 0,0025.
- Var skeptisk tills resultatet har **upprepats** i en ny studie.

## 15.3 Fälla 2: Signifikant men oviktigt

Ett nätbaserat intelligenstest har gjorts av två miljoner personer. Rubrik: *"Statistiskt säkerställt: kaffedrickare smartare!"*

In [ ]:
n_per_group = 1_000_000        # ← ändra: pröva 100 och 10_000
true_difference = 0.3          # IQ-poäng (standardavvikelsen är 15)

coffee = rng.normal(100 + true_difference, 15, n_per_group)
no_coffee = rng.normal(100, 15, n_per_group)

res = stats.ttest_ind(coffee, no_coffee, equal_var=False)
ci = res.confidence_interval(0.95)
print(f"Medel kaffe: {coffee.mean():.2f}   medel ej kaffe: {no_coffee.mean():.2f}")
print(f"p-värde: {res.pvalue:.2g}")
print(f"95 % KI för skillnaden: [{ci.low:.2f}, {ci.high:.2f}] IQ-poäng")

p-värdet är extremt litet. Men titta på konfidensintervallet: skillnaden är en bråkdel av en IQ-poäng, på en skala där
standardavvikelsen är 15. Så här ser de två fördelningarna ut:

In [ ]:
x = np.linspace(40, 160, 400)
plt.plot(x, stats.norm.pdf(x, 100, 15), label="Ej kaffe")
plt.plot(x, stats.norm.pdf(x, 100 + true_difference, 15), linestyle="--", label="Kaffe")
plt.xlabel("IQ")
plt.ylabel("Täthet")
plt.title("Två grupper med 'statistiskt säkerställd' skillnad")
plt.legend()
plt.show()

Kurvorna går knappt att skilja åt. **p-värdet mäter hur säkra vi är på att skillnaden inte är noll, inte hur stor den är.**
Med tillräckligt mycket data blir varje skillnad signifikant, hur liten den än är.

Ett sätt att bedöma storleken är att uttrycka skillnaden i standardavvikelser (*effektstorlek*): 0,3 / 15 = 0,02.
I notebook 10 kallade vi 0,2 en liten effekt. Det här är en tiondel av det.

(Och även om skillnaden vore stor: det här är en observationsstudie med självvalda deltagare. Vilka fler frågor ur checklistan borde ställas?)

## 15.4 Fälla 3: "Ingen signifikant skillnad"

Den omvända fällan. Rubrik: *"Studie: ny undervisningsmetod ger ingen effekt"*. I studien deltog 12 elever per grupp.

In [ ]:
group_size = 12          # ← ändra: pröva 50 och 200
true_effect = 6          # poäng – en verklig förbättring (sd = 12, alltså en halv standardavvikelse)

rng_study = np.random.default_rng(2)        # eget frö, så att cellen ger samma data varje gång
control = rng_study.normal(65, 12, group_size)
treated = rng_study.normal(65 + true_effect, 12, group_size)

res = stats.ttest_ind(treated, control, equal_var=False)
ci = res.confidence_interval(0.95)
print(f"Skillnad i medelpoäng: {treated.mean() - control.mean():.1f}")
print(f"p-värde: {res.pvalue:.3f}")
print(f"95 % KI för skillnaden: [{ci.low:.1f}, {ci.high:.1f}] poäng")

Testet är inte signifikant. Men konfidensintervallet berättar den verkliga historien: det sträcker sig från en
*försämring* på några poäng till en mycket stor *förbättring*. Studien kan helt enkelt inte avgöra saken.

> **"Vi fann ingen signifikant skillnad"** betyder *"vi kunde inte påvisa någon skillnad"*.
> Det betyder inte *"vi har visat att det inte finns någon skillnad"*.
> Ett brett konfidensintervall som innehåller både noll och stora effekter betyder: **vi vet inte**.

✏️ **Din tur:** Hur stor styrka hade studien? Simulera 2 000 studier med 12 elever per grupp och en sann effekt på 6 poäng
(sd = 12), och räkna andelen som blir signifikanta. (Samma teknik som i notebook 10.)

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
control_sim = rng.normal(65, 12, (2000, 12))
treated_sim = rng.normal(71, 12, (2000, 12))
p_sim = stats.ttest_ind(treated_sim, control_sim, axis=1, equal_var=False).pvalue
print(f"Styrka: {(p_sim < 0.05).mean():.0%}")
# Ungefär 20 %. Studien hade missat en verklig effekt fyra gånger av fem.
```

</details>

## 15.5 Slutuppgift: granska ett påstående

Ett företag bakom en träningsapp skickar ut ett pressmeddelande:

> **"Användare av vår app sover bättre! En undersökning med 400 personer visar att appanvändare sover i genomsnitt
> längre än andra. Skillnaden är statistiskt säkerställd."**

Du har fått tillgång till datan. Varje rad är en person: ålder, om personen använder appen och sömn i timmar per natt.

In [ ]:
n = 400
rng_app = np.random.default_rng(43)
age = rng_app.integers(20, 71, n)
uses_app = rng_app.random(n) < np.clip(1.05 - age / 60, 0.05, 0.95)
sleep = 9.0 - 0.035 * age + rng_app.normal(0, 0.8, n)

study = pd.DataFrame({"age": age, "uses_app": uses_app, "sleep": sleep.round(1)})
study.head()

✏️ **Din tur:** Arbeta dig igenom stegen. Försök själv innan du tittar på lösningsförslaget efter varje steg.

**Steg 1 – Kontrollera påståendet.** Jämför sömnen mellan appanvändare och övriga: medelvärden, lådagram,
t-test och konfidensintervall för skillnaden. Stämmer det företaget säger?

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
users = study[study["uses_app"]]
non_users = study[~study["uses_app"]]

plt.boxplot([users["sleep"], non_users["sleep"]])
plt.xticks([1, 2], ["Använder appen", "Använder inte appen"])
plt.ylabel("Sömn (timmar per natt)")
plt.show()

res_app = stats.ttest_ind(users["sleep"], non_users["sleep"], equal_var=False)
ci_app = res_app.confidence_interval(0.95)
print(f"Antal: {len(users)} användare, {len(non_users)} övriga")
print(f"Medelsömn: {users['sleep'].mean():.2f} mot {non_users['sleep'].mean():.2f} timmar")
print(f"p-värde: {res_app.pvalue:.2g}   95 % KI för skillnaden: [{ci_app.low:.2f}, {ci_app.high:.2f}] timmar")
# Ja: appanvändarna sover längre, och skillnaden är statistiskt säkerställd. Siffrorna i pressmeddelandet stämmer.
```

</details>

**Steg 2 – Leta efter en bakomliggande variabel.** Företaget antyder att appen *ger* bättre sömn. Gå igenom checklistan.
Skiljer sig grupperna åt i något annat avseende? Undersök åldern i de två grupperna, och sambandet mellan ålder och sömn
(spridningsdiagram och korrelation).

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
print(f"Medelålder: {users['age'].mean():.1f} år bland användare, {non_users['age'].mean():.1f} år bland övriga")

r_age, p_age = stats.pearsonr(study["age"], study["sleep"])
plt.scatter(non_users["age"], non_users["sleep"], s=10, label="Använder inte appen")
plt.scatter(users["age"], users["sleep"], s=10, label="Använder appen")
plt.xlabel("Ålder (år)")
plt.ylabel("Sömn (timmar per natt)")
plt.title(f"Ålder och sömn: r = {r_age:.2f}")
plt.legend()
plt.show()
# Appanvändarna är mycket yngre, och yngre personer sover längre. Ålder är en möjlig bakomliggande variabel.
```

</details>

**Steg 3 – Jämför lika med lika.** Om appen har en egen effekt borde appanvändare sova längre än andra *i samma ålder*.
Dela in personerna i åldersgrupper med `pd.cut(study["age"], bins=[19, 30, 40, 50, 60, 70])` och jämför medelsömnen för
användare och övriga inom varje grupp. Ledtråd: `study.groupby([age_group, "uses_app"])["sleep"].mean().unstack()`.

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
age_group = pd.cut(study["age"], bins=[19, 30, 40, 50, 60, 70])
comparison = study.groupby([age_group, "uses_app"], observed=True)["sleep"].agg(["mean", "count"]).unstack()
display(comparison.round(2))

for group, part in study.groupby(age_group, observed=True):
    res_group = stats.ttest_ind(part[part["uses_app"]]["sleep"], part[~part["uses_app"]]["sleep"], equal_var=False)
    print(f"{group}: p = {res_group.pvalue:.2f}")
# Inom åldersgrupperna är skillnaderna små, går åt olika håll och är inte signifikanta.
```

</details>

**Steg 4 – Skriv din bedömning.** Formulera tre–fyra meningar: vad visar datan, vad visar den inte, och vad skulle krävas
för att belägga företagets påstående?

<details>
<summary>💡 <b>Förslag till bedömning</b></summary>

Det stämmer att appanvändarna i undersökningen sover längre i genomsnitt, och skillnaden är statistiskt säkerställd.
Men användarna är betydligt yngre än de övriga, och sömnlängden minskar med åldern. När personer i samma ålder jämförs
finns ingen tydlig skillnad kvar. Sambandet förklaras alltså av en bakomliggande variabel (ålder), och datan ger inget stöd för
att appen *orsakar* bättre sömn.

Dessutom: vi vet inte hur de 400 personerna valdes ut eller hur stort bortfallet var, och sömnen är troligen självrapporterad (mätfel).
För att belägga en effekt krävs ett **experiment**, där det lottas vilka som får använda appen.

(Facit: datan simulerades så att sömnen beror på åldern och inte alls på appen.)

</details>

Lägg märke till vad som hände: **varje siffra i pressmeddelandet var korrekt**, och slutsatsen var ändå fel.
Felet låg inte i räkningarna utan i tolkningen. Det är därför den kritiska värderingen är den viktigaste delen av statistiken.

## Kursen i en mening per notebook

| | |
|---|---|
| 1 | Notebooks är till för att experimentera. |
| 2 | Variabelns typ avgör vad man får räkna ut. |
| 3 | Beskriv data med läge, spridning och en bild. |
| 4 | En fördelning säger hur vanligt varje värde är. Sannolikhet är area. |
| 5 | En skattning varierar mellan stickprov, med ett medelfel som krymper som 1/√n. |
| 6 | Mer data botar slumpfel men inte bias. |
| 7 | Ett konfidensintervall är skattning ± felmarginal, och 95 % syftar på metoden. |
| 8 | p-värdet säger hur ovanlig datan vore om nollhypotesen var sann. |
| 9 | Signifikant är inte samma sak som viktigt. |
| 10 | Tester kan larma falskt och missa verkliga effekter. Styrkan avgörs av stickprovets och effektens storlek. |
| 11 | Chi-två-testet jämför det observerade med det som väntas utan samband. |
| 12 | Korrelation mäter linjärt samband, inte orsak. |
| 13 | Regression beskriver sambandet med en linje. Extrapolera inte. |
| 14 | Statistiken är densamma i R, bara stavningen skiljer. |
| 15 | Rätt uträknat är inte samma sak som rätt tolkat. |

### Avslutande kontrollfrågor
1. En tidning skriver: "Ny studie: de som äter frukost har 20 % lägre risk för hjärtsjukdom." Vilka tre frågor ställer du först?
2. En kollega har testat 40 olika kundsegment och hittat två där kampanjen gav "signifikant effekt" (p < 0,05). Kommentera.
3. En rapport redovisar "p = 0,049" för huvudresultatet och ingenting annat. Vad saknar du?

<details>
<summary>💡 <b>Svar</b></summary>

1. Till exempel: Är det ett experiment eller en observationsstudie (kan frukostätare skilja sig åt i annat, som motion, rökning och inkomst)?
   Hur stort var stickprovet och hur brett är konfidensintervallet? 20 % lägre än vad – hur stor är risken i absoluta tal?
2. Med 40 tester och α = 0,05 väntar man sig ungefär två falska larm även om kampanjen inte har någon effekt alls.
   Resultatet är precis vad slumpen skulle ge. Testa om de två segmenten i en ny omgång.
3. Effektens storlek och ett konfidensintervall, stickprovets storlek, hur urvalet gjordes, om hypotesen bestämdes i förväg
   och hur många andra tester som gjordes. Ett p-värde strax under 0,05 utan sammanhang är svagt belägg.

</details>

**Bra jobbat – du är klar med kursen!** Gå gärna tillbaka till [kursöversikten](../README.md), eller öppna
`statistik_grunder.ipynb` i projektets rot, som samlar alla begrepp i en enda notebook för repetition.